# Band Padding Ablation Analysis

Compare zero-filled channel padding (full S2 input) against the standard baseline (BandFilter only).

- **Padding**: encoder receives all S2 bands; missing ones are zero-filled after normalization
- **Baseline**: standard UPerNet runs from `pangaea_full_seg/upernet/` (BandFilter, no padding)

In [1]:
import re
import os
from pathlib import Path
import pandas as pd
import numpy as np

EXPERIMENTS_BASE = Path("/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/niklas/experiments")

# _s2full variants (band padding experiment) map to same display name as base encoder
ENCODER_LABELS = {
    "dofa_s2full":               "DOFA",
    "dofav2_s2full":             "DOFA+",
    "panopticon_s2full":         "Panopticon",
    "smarties_vitb_s2full":      "SMARTIES",
    "copernicus_fm_base_s2full": "Copernicus-FM",
    "carl_no_internal_norm":     "CARL",
    "carl_imagewise":            "CARL",
    "dofa":               "DOFA",
    "dofav2":             "DOFA+",
    "carl":               "CARL",
    "panopticon":         "Panopticon",
    "copernicus_fm_base": "Copernicus-FM",
    "smarties_vitb":      "SMARTIES",
    "dinov2_vitb14":      "DINOv2",
    "unet_encoder":       "UNet",
    "unet_encoder_mi":    "UNet",
    "vit_scratch":        "ViT-Scratch",
    "vit_scratch_mi":     "ViT-Scratch",
}

DATASET_LABELS = {
    "hlsburnscars":      "HLS-BurnScars",
    "sen1floods11":      "Sen1Floods11",
    "mados":             "MADOS",
    "pastis":            "PASTIS",
    "dynamicen":         "DynamicEarthNet",
    "croptypemapping":   "CropTypeMapping",
    "spacenet7cd":       "SpaceNet7-CD",
    "ai4smallfarms":     "AI4SmallFarms",
    "fivebillionpixels": "FiveBillionPixels",
}

DATASET_ORDER = list(DATASET_LABELS.keys())

# _s2full variants and carl_imagewise/carl_no_internal_norm must appear before
# their shorter counterpart ("carl") for greedy matching -- otherwise e.g.
# "carl_imagewise_..." matches the shorter "carl" prefix, leaving "imagewise_..."
# unconsumed and making decoder detection fail (metric value stays correct, but
# decoder ends up blank).
KNOWN_ENCODERS = [
    "copernicus_fm_base_s2full", "copernicus_fm_base", "copernicus_fm_large",
    "smarties_vitb_s2full", "smarties_vitb", "dinov2_vitb14",
    "panopticon_s2full",
    "dofav2_s2full", "dofa_s2full",
    "dofav2", "dofa", "panopticon",
    "carl_no_internal_norm", "carl_imagewise", "carl",
    "vit_scratch_mi", "vit_scratch",
    "unet_encoder_mi", "unet_encoder",
]

KNOWN_DECODERS = [
    "seg_upernet_mt_ltae", "seg_upernet",
    "seg_siamupernet_conc", "seg_siamupernet_diff",
    "seg_siam_mlp_fusion", "seg_mlp_fusion",
    "seg_siamunet_conc", "seg_siamunet_diff", "seg_unet",
    "seg_linear_probe", "seg_siam_linear_probe",
    "cls_linear_probe",
]

DATASET_RE = re.compile(
    r"_(" + "|".join(re.escape(d) for d in DATASET_ORDER) + r")_"
)

# Ablation encoders in display order
ABL_ENCODERS_DISPLAY = ["DOFA", "DOFA+", "CARL", "Panopticon", "SMARTIES", "Copernicus-FM"]

print("Config loaded.")

Config loaded.


In [2]:
def parse_run_dir_name(name: str) -> dict:
    """Extract dataset, encoder, decoder from a run directory name."""
    m = DATASET_RE.search(name)
    if not m:
        return {}
    dataset = m.group(1)
    rest = name[m.end():]

    # DynamicEarthNet true-wavelength (B8A) reruns insert "true_wl_" between
    # dataset and encoder -- strip it before encoder matching or detection
    # silently fails and the run falls into a blank-encoder row.
    for tag in ("true_wl_",):
        if rest.startswith(tag):
            rest = rest[len(tag):]
            break

    encoder = next((e for e in KNOWN_ENCODERS if rest.startswith(e + "_")), None)
    if encoder:
        rest = rest[len(encoder) + 1:]

    decoder = next((d for d in KNOWN_DECODERS if rest.startswith(d)), None)

    return {"dataset": dataset, "encoder": encoder or "", "decoder": decoder or ""}


def parse_log_metrics(log_path: Path) -> dict:
    """Parse train.log-0 for test metrics (last test block)."""
    if not log_path.exists():
        return {}

    text = log_path.read_text(errors="ignore")
    sections = re.split(r"\[test\] ------- IoU", text)
    if len(sections) < 2:
        return {}
    test_block = "[test] ------- IoU" + sections[-1]

    metrics = {}
    metric_map = {
        "IoU":       "test_mIoU",
        "F1-score":  "test_mF1",
        "Precision": "test_mPrecision",
        "Recall":    "test_mRecall",
    }
    for section_name, metric_key in metric_map.items():
        pattern = rf"\[test\] ------- {section_name}.*?\[test\] Mean\s+([\d.]+)"
        m = re.search(pattern, test_block, re.DOTALL)
        if m:
            metrics[metric_key] = float(m.group(1))
    return metrics


def load_experiment_folder(
    folder,
    encoder_map=None,
    dataset_map=None,
    metric="test_mIoU",
    keep_all_metrics=False,
):
    """
    Load all runs from an experiment folder and return a long-form DataFrame.
    Walks up to 2 levels deep to find run dirs containing train.log-0.
    """
    enc_map = {**ENCODER_LABELS, **(encoder_map or {})}
    ds_map  = {**DATASET_LABELS,  **(dataset_map  or {})}
    folder  = Path(folder)

    run_dirs = []
    for child in sorted(folder.iterdir()):
        if not child.is_dir():
            continue
        if (child / "train.log-0").exists():
            run_dirs.append(child)
        else:
            for grandchild in sorted(child.iterdir()):
                if grandchild.is_dir() and (grandchild / "train.log-0").exists():
                    run_dirs.append(grandchild)

    rows = []
    for run_dir in run_dirs:
        parsed = parse_run_dir_name(run_dir.name)
        if not parsed.get("dataset"):
            continue
        log_metrics = parse_log_metrics(run_dir / "train.log-0")
        if not log_metrics:
            continue

        enc_raw = parsed["encoder"]
        ds_raw  = parsed["dataset"]

        row = {
            "run_dir":     run_dir.name,
            "encoder_raw": enc_raw,
            "decoder":     parsed["decoder"],
            "dataset_raw": ds_raw,
            "encoder":     enc_map.get(enc_raw, enc_raw),
            "dataset":     ds_map.get(ds_raw, ds_raw),
        }
        if keep_all_metrics:
            row.update(log_metrics)
        elif metric in log_metrics:
            row[metric] = log_metrics[metric]
        else:
            continue
        rows.append(row)

    return pd.DataFrame(rows)


def to_pivot(
    df,
    metric="test_mIoU",
    index="encoder",
    columns="dataset",
    aggfunc="last",
    dataset_order=None,
    encoder_order=None,
    add_mean=True,
):
    """Pivot a long-form results DataFrame to encoder x dataset."""
    if metric not in df.columns:
        raise ValueError(f"Metric '{metric}' not in DataFrame: {list(df.columns)}")

    pivot = df.pivot_table(index=index, columns=columns, values=metric, aggfunc=aggfunc)

    if dataset_order is None:
        ordered_ds = [DATASET_LABELS[d] for d in DATASET_ORDER if DATASET_LABELS[d] in pivot.columns]
        remaining  = [c for c in pivot.columns if c not in ordered_ds]
        dataset_order = ordered_ds + remaining
    pivot = pivot.reindex(columns=[c for c in dataset_order if c in pivot.columns])

    if encoder_order is not None:
        pivot = pivot.reindex([e for e in encoder_order if e in pivot.index])

    if add_mean:
        pivot["Mean"] = pivot.mean(axis=1, skipna=True).round(3)

    return pivot.round(2)


print("Functions defined.")

Functions defined.


## Band Padding Results

Zero-fills missing S2 channels so every encoder receives the full S2 band set.

In [3]:
df_pad_raw = load_experiment_folder(
    EXPERIMENTS_BASE / "ablations" / "band_padding",
    metric="test_mIoU",
)
# CARL: keep only the carl_imagewise rerun (carl_imagewise_band_pad/), drop the
# older carl_no_internal_norm rows (carl_band_pad/) -- CARL's baseline below
# uses carl_imagewise (its main-benchmark default), so the padding side needs
# to match it for a clean, isolated padding effect, same as every other
# encoder where baseline and padding share one encoder config.
n_before = len(df_pad_raw)
df_pad = df_pad_raw[df_pad_raw["encoder_raw"] != "carl_no_internal_norm"].copy()
print(f"{len(df_pad)} runs loaded from band_padding/ "
      f"({n_before - len(df_pad)} old carl_no_internal_norm rows excluded)")
# aggfunc="last" (not "max"): SpaceNet7-CD and DynamicEarthNet each have an old
# run (dice-loss / B5-mislabeled, June) and a fixed rerun (weighted_bce_sqrt /
# B8A, September) sitting side by side in these folders -- "last" picks the
# fixed rerun since run dirs are timestamp-prefixed and sorted chronologically;
# "max" would silently prefer whichever number happens to be numerically larger,
# which is the stale run for several encoders (e.g. old CARL SpaceNet7-CD 57.51
# vs fixed 55.11). Matches the aggfunc used by to_pivot() below.
df_pad.pivot_table(index="encoder", columns="dataset", values="test_mIoU", aggfunc="last")

55 runs loaded from band_padding/ (10 old carl_no_internal_norm rows excluded)


dataset,AI4SmallFarms,CropTypeMapping,DynamicEarthNet,FiveBillionPixels,HLS-BurnScars,MADOS,PASTIS,Sen1Floods11,SpaceNet7-CD
encoder,,,,,,,,,
Copernicus-FM,33.254,52.649,37.122,57.308,82.218,59.281,37.153,89.703,58.341
DOFA,35.365,54.383,38.724,70.161,81.666,58.438,28.664,89.276,62.420
DOFA+,37.653,49.123,43.737,70.023,78.698,46.662,22.187,88.052,62.010
Panopticon,38.152,49.406,43.349,68.840,83.826,61.629,36.857,90.298,62.958
SMARTIES,34.927,37.077,37.231,58.027,82.960,62.633,34.030,90.120,59.540


In [4]:
pivot_pad = to_pivot(df_pad, encoder_order=ABL_ENCODERS_DISPLAY)
pivot_pad

dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean
encoder,,,,,,,,,,
DOFA,81.67,89.28,58.44,28.66,38.72,54.38,62.42,35.36,70.16,57.68
DOFA+,78.70,88.05,46.66,22.19,43.74,49.12,62.01,37.65,70.02,55.35
Panopticon,83.83,90.30,61.63,36.86,43.35,49.41,62.96,38.15,68.84,59.48
SMARTIES,82.96,90.12,62.63,34.03,37.23,37.08,59.54,34.93,58.03,55.17
Copernicus-FM,82.22,89.70,59.28,37.15,37.12,52.65,58.34,33.25,57.31,56.34


## Baseline (Standard UPerNet, BandFilter only)

Loaded from `pangaea_full_seg/upernet/`, filtered to the 6 ablation encoders.

In [5]:
df_baseline_scan = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet",
    metric="test_mIoU",
)
# Keep only encoders also present in the padding experiment. CARL is excluded
# from this generic scan and loaded separately below: pangaea_full_seg/upernet/
# holds CARL's runs in *two* different subfolders (carl_seg/ = plain "carl",
# carl_imagewise_seg/ = the encoder's own per-image normalization) for the same
# 3 datasets, and since aggfunc="last" resolves ties by folder-iteration order
# (alphabetical: "carl_imagewise_seg" < "carl_seg"), it was silently picking
# carl_seg's plain-carl runs, not carl_imagewise -- CARL's actual benchmark
# default (per-image norm, matching the CARL paper), which is also what the
# normalization-ablation notebook uses as CARL's baseline.
df_baseline = df_baseline_scan[
    df_baseline_scan["encoder"].isin([e for e in ABL_ENCODERS_DISPLAY if e != "CARL"])
].copy()

df_baseline_carl = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet" / "carl_imagewise_seg",
    metric="test_mIoU",
)
if not df_baseline_carl.empty:
    df_baseline_carl = df_baseline_carl[df_baseline_carl["encoder"] == "CARL"].copy()
    df_baseline = pd.concat([df_baseline, df_baseline_carl], ignore_index=True)

print(f"{len(df_baseline)} runs loaded from upernet/ (ablation encoders only; CARL = carl_imagewise)")
pivot_baseline = to_pivot(df_baseline, encoder_order=ABL_ENCODERS_DISPLAY)
pivot_baseline

66 runs loaded from upernet/ (ablation encoders only; CARL = carl_imagewise)


dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean
encoder,,,,,,,,,,
DOFA,80.64,87.69,60.39,30.37,39.59,36.21,62.12,36.44,69.93,55.93
DOFA+,78.27,88.01,48.60,19.79,43.91,36.23,63.45,38.08,71.22,54.17
CARL,83.94,89.02,46.63,39.50,37.75,58.80,58.86,31.66,47.59,54.86
Panopticon,84.22,90.59,68.22,36.81,44.74,47.99,63.48,37.88,70.10,60.45
SMARTIES,82.84,90.05,62.28,35.75,40.67,52.04,58.85,35.47,57.32,57.25
Copernicus-FM,82.33,89.11,56.90,33.78,36.47,48.08,61.95,33.35,62.80,56.09


## Δ Padding − Baseline

Positive values mean padding improved performance; negative means it hurt.

In [6]:
# Align columns (datasets present in both)
shared_cols = [c for c in pivot_pad.columns if c in pivot_baseline.columns and c != "Mean"]
delta = (pivot_pad[shared_cols] - pivot_baseline[shared_cols]).round(3)

# Reindex to consistent encoder order
delta = delta.reindex([e for e in ABL_ENCODERS_DISPLAY if e in delta.index])
delta["Mean Δ"] = delta.mean(axis=1, skipna=True).round(3)

# Highlight: green = positive, red = negative
def color_delta(val):
    if pd.isna(val):
        return ""
    color = "#c6efce" if val > 0 else ("#ffc7ce" if val < 0 else "")
    return f"background-color: {color}"

delta.style.applymap(color_delta)

/dss/dsstbyfs02/scratch/08/di54zop/di54zop/ipykernel_117977/1732051092.py:16: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  delta.style.applymap(color_delta)


dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean Δ
encoder,,,,,,,,,,
DOFA,1.030000,1.590000,-1.950000,-1.710000,-0.870000,18.170000,0.300000,-1.080000,0.230000,1.746000
DOFA+,0.430000,0.040000,-1.940000,2.400000,-0.170000,12.890000,-1.440000,-0.430000,-1.200000,1.176000
CARL,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
Panopticon,-0.390000,-0.290000,-6.590000,0.050000,-1.390000,1.420000,-0.520000,0.270000,-1.260000,-0.967000
SMARTIES,0.120000,0.070000,0.350000,-1.720000,-3.440000,-14.960000,0.690000,-0.540000,0.710000,-2.080000
Copernicus-FM,-0.110000,0.590000,2.380000,3.370000,0.650000,4.570000,-3.610000,-0.100000,-5.490000,0.250000


In [7]:
# Summary: how many datasets improved vs degraded per encoder
summary = pd.DataFrame({
    "Improved (Δ > 0)": (delta[shared_cols] > 0).sum(axis=1),
    "Degraded (Δ < 0)": (delta[shared_cols] < 0).sum(axis=1),
    "Neutral (Δ = 0)": (delta[shared_cols] == 0).sum(axis=1),
    "Mean Δ":           delta["Mean Δ"],
})
summary

,Improved (Δ > 0),Degraded (Δ < 0),Neutral (Δ = 0),Mean Δ
encoder,,,,
DOFA,5,4,0,1.746
DOFA+,4,5,0,1.176
CARL,0,0,0,NaN
Panopticon,3,6,0,-0.967
SMARTIES,5,4,0,-2.080
Copernicus-FM,5,4,0,0.250


In [8]:
# Check which runs are still missing in the padding experiment
# Reindex against the full expected encoder list first -- to_pivot() only
# reindexes to encoders that already have >=1 row, so an encoder with *zero*
# data (e.g. CARL while its carl_imagewise rerun is still pending) has no row
# at all rather than a row of NaNs, and would otherwise slip past the isna()
# check below undetected.
print("=== Missing padding runs ===")
pivot_pad_full = pivot_pad.drop(columns="Mean").reindex(ABL_ENCODERS_DISPLAY)
missing = pivot_pad_full.isna().stack()
missing_rows = missing[missing].reset_index()[["encoder", "dataset"]]
if missing_rows.empty:
    print("None — all runs completed.")
else:
    print(missing_rows.to_string(index=False))

=== Missing padding runs ===
encoder           dataset
   CARL     HLS-BurnScars
   CARL      Sen1Floods11
   CARL             MADOS
   CARL            PASTIS
   CARL   DynamicEarthNet
   CARL   CropTypeMapping
   CARL      SpaceNet7-CD
   CARL     AI4SmallFarms
   CARL FiveBillionPixels


## Paper-style table: w/o padding (baseline) vs w/ padding

In [9]:
def color_diff_cell(val):
    if not isinstance(val, str) or val == "—":
        return ""
    v = float(val)
    if v > 0:
        return "color: #276221; font-weight: bold"
    if v < 0:
        return "color: #9c1e1e; font-weight: bold"
    return ""


# Datasets present in both pivots (excluding Mean column)
datasets = [c for c in pivot_baseline.columns if c in pivot_pad.columns and c != "Mean"]
base = pivot_baseline[datasets]
pad  = pivot_pad[datasets]

cols, data = [], {}
for ds in datasets:
    diff = (pad[ds] - base[ds]).round(2)
    data[(ds, "w/o padding")] = base[ds].round(2)
    data[(ds, "w/ padding")]  = pad[ds].round(2)
    data[(ds, "Δ")]           = diff.apply(lambda v: f"{v:+.2f}" if not pd.isna(v) else "—")
    cols += [(ds, "w/o padding"), (ds, "w/ padding"), (ds, "Δ")]

tbl = pd.DataFrame(data, columns=pd.MultiIndex.from_tuples(cols))
tbl = tbl.reindex([e for e in ABL_ENCODERS_DISPLAY if e in tbl.index])

diff_cols = [c for c in tbl.columns if c[1] == "Δ"]
display(
    tbl.style
       .map(color_diff_cell, subset=diff_cols)
       .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)